# Complementary telescope-resource evidence — GRANDMA reference & ICARE semantics (A2)

This notebook sits between `A_eda.ipynb` and `B_decisions.ipynb`. It gathers
two narrowly scoped pieces of evidence that `A_eda.ipynb` could not resolve
from the ICARE tables alone:

1. an external reference — the GRANDMA telescope-network table supplied for
   this project — compared diagnostically against ICARE telescope names, to
   see what it could potentially add where ICARE is sparse (footprint/FOV,
   sensitivity, filters, aperture, robotic status);
2. authoritative-source answers (SkyPortal source code) for two ICARE
   ambiguities `A_eda.ipynb` flagged: the `morning`/`evening` mixed
   representation, and allocation semantics including the nested
   allocation id 78 discrepancy.

It is **descriptive only**. No ICARE↔GRANDMA name mapping is persisted,
no data is merged, and no source precedence is decided here. Its output is
evidence for the questions listed at the end, not answers to them.


In [1]:
import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/telescopes").is_dir())
CAPTURE_ID = "capture_20260808_071334"
ICARE_INTERIM_DIR = ROOT / "data/interim/telescopes" / CAPTURE_ID
ICARE_RAW_DIR = ROOT / "data/raw/telescopes/icare" / CAPTURE_ID
GRANDMA_RAW_DIR = ROOT / "data/raw/reference/grandma"
GRANDMA_INTERIM_PATH = ROOT / "data/interim/telescopes/reference/grandma_table.parquet"

ICARE_TABLE_NAMES = ["telescopes", "instruments", "allocations", "observations"]
ICARE = {name: pd.read_parquet(ICARE_INTERIM_DIR / f"{name}.parquet") for name in ICARE_TABLE_NAMES}
GRANDMA = pd.read_parquet(GRANDMA_INTERIM_PATH)

for name, frame in ICARE.items():
    print(f"ICARE {name:14s} rows={len(frame):3d} columns={frame.shape[1]:3d}")
print(f"GRANDMA table      rows={len(GRANDMA):3d} columns={GRANDMA.shape[1]:3d}  <- {GRANDMA_INTERIM_PATH}")


def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Repository-safety baseline: hashed now, re-checked in the final cell.
ICARE_FILES_TO_WATCH = sorted(ICARE_RAW_DIR.glob("*.json")) + sorted(ICARE_INTERIM_DIR.glob("*.parquet"))
HASHES_BEFORE = {str(p): sha256_of(p) for p in ICARE_FILES_TO_WATCH}
print(f"\nwatching {len(HASHES_BEFORE)} ICARE raw/interim files for accidental modification")


def has_content(series):
    """True where a cell holds real content: '', '-', '[]', '{}', null and NaN are empty."""
    empty_tokens = {"", "-", "[]", "{}"}
    def alive(value):
        if value is None:
            return False
        if isinstance(value, str):
            return value.strip() not in empty_tokens
        if isinstance(value, float) and np.isnan(value):
            return False
        return True
    return series.map(alive)


def clip(value, limit=60):
    text = str(value)
    return f"{text[:limit]}..." if len(text) > limit else text


DECISION_QUESTIONS = []


def note_question(question, evidence, fields, scope):
    DECISION_QUESTIONS.append({"question": question, "evidence": evidence,
                               "relevant_table_fields": fields, "measured_scope": scope})


ICARE telescopes     rows= 89 columns= 21
ICARE instruments    rows= 95 columns= 42
ICARE allocations    rows= 38 columns= 50
ICARE observations   rows= 93 columns= 27
GRANDMA table      rows= 39 columns= 15  <- /home/meneses/project_astronomical/MAFORAI/data/interim/telescopes/reference/grandma_table.parquet

watching 9 ICARE raw/interim files for accidental modification


## GRANDMA source provenance

Identification, established directly from the supplied document itself
(never from its filename): the exact document, where it is preserved, and
how it was extracted. Elements the document does not state are recorded
as `NOT ESTABLISHED` rather than guessed.


In [2]:
with (GRANDMA_RAW_DIR / "manifest.json").open() as handle:
    grandma_source = json.load(handle)

for key in ["source_file", "sha256", "page_count", "title_english", "title_french", "author",
           "document_type", "institution", "defense_location", "defense_date_day_month",
           "defense_year", "doi", "manuscript_status", "relevant_table", "relevant_table_title",
           "relevant_table_pages", "relevant_table_temporal_scope_statement", "extraction_method"]:
    print(f"{key:42s}: {grandma_source.get(key)}")

preserved_pdf = ROOT / grandma_source["source_file"]
actual_sha256 = sha256_of(preserved_pdf)
print(f"\npreserved PDF sha256 matches manifest: {actual_sha256 == grandma_source['sha256']}")
print(f"GRANDMA table rows extracted: {len(GRANDMA)} (39 telescope rows found on pages "
      f"{grandma_source['relevant_table_pages']})")


source_file                               : data/raw/reference/grandma/Second_Phd_Thesis____Sarah-7.pdf
sha256                                    : 3c95bc9a16e62baa2e6d357a93c0d6743a31e2e8db685ad57b52bf9bad3fb3d9
page_count                                : 85
title_english                             : Multi-messenger Astronomy with GRANDMA
title_french                              : L'Astronomie Multi-messagers avec GRANDMA
author                                    : Sarah Antier
document_type                             : Habilitation à diriger des recherches (HDR) manuscript
institution                               : Université Paris-Saclay
defense_location                          : Orsay
defense_date_day_month                    : 9 October
defense_year                              : NOT ESTABLISHED (not printed on the title page; PDF CreationDate metadata is 2026-07-21 but that reflects file generation, not a stated authorial date)
doi                                       : NOT

In [3]:
print("Table caption, quoted verbatim:")
print(f'  "{grandma_source["relevant_table_caption"]}"')


Table caption, quoted verbatim:
  "Table 1.1: GRANDMA telescope network. "MNight" corresponds to the time zone of maximum night time. Mlim corresponds to the typical maximum limiting mag obtained in less than 1 h. The "Use" colon include "VF" for very frequent observations for GRANDMA, "F" as Frequent, "R" as Rare, "VR" as very Rare, and "nOP" as not Operational. The column "Rob" include if the telescope is robotic or not."


## GRANDMA field semantics: SOURCE-DEFINES vs INFERRED

Read directly from the table's own caption (quoted above) and the
surrounding body text. Unlike the reference previously (incorrectly) used
for this stage, THIS document's Table 1.1 literally carries every column
named in the Stage 4 brief: Name, Location, MNight, Size, FOV, Filter,
Mlim, Use, Rob.


In [4]:
semantics_rows = [
    {"field": "Name", "classification": "SOURCE-DEFINES",
     "documented_meaning": "The telescope/instrument identifier used throughout the network table."},
    {"field": "Location", "classification": "SOURCE-DEFINES",
     "documented_meaning": "Observatory name (free text), not coordinates."},
    {"field": "MNight", "classification": "SOURCE-DEFINES",
     "documented_meaning": "Caption, verbatim: \"'MNight' corresponds to the time zone of maximum "
                            "night time.\" A UTC scheduling window (e.g. '10-16h'), not a night count."},
    {"field": "Size", "classification": "SOURCE-DEFINES",
     "documented_meaning": "Telescope aperture in metres (column header '(m)'). '-' for the "
                            "spectroscopy-section rows that report no aperture."},
    {"field": "FOV", "classification": "SOURCE-DEFINES",
     "documented_meaning": "Field of view in degrees (column header '(deg)'), width x height or a "
                            "single circular value. '-' throughout the spectroscopy section."},
    {"field": "Filter", "classification": "SOURCE-DEFINES / partially INFERRED",
     "documented_meaning": "Photometric filter codes for the Photometry-section rows (e.g. 'BVRCIC'; "
                            "SOURCE-DEFINES). For the Spectroscopy-section rows the same column "
                            "position instead holds a wavelength range in what is almost certainly "
                            "Angstrom (e.g. '3200-7000'), which the caption never states explicitly -- "
                            "the unit is INFERRED from domain convention, not confirmed by the text."},
    {"field": "Mlim", "classification": "SOURCE-DEFINES",
     "documented_meaning": "Caption, verbatim: \"Mlim corresponds to the typical maximum limiting "
                            "mag obtained in less than 1 h.\" A typical/nominal depth bounded by a "
                            "1-hour exposure ceiling -- not a guaranteed or per-observation value."},
    {"field": "Use", "classification": "SOURCE-DEFINES",
     "documented_meaning": "Caption, verbatim: \"The 'Use' colon include 'VF' for very frequent "
                            "observations for GRANDMA, 'F' as Frequent, 'R' as Rare, 'VR' as very "
                            "Rare, and 'nOP' as not Operational.\" ('colon' appears to be a typo for "
                            "'column' in the source; quoted as printed.)"},
    {"field": "Rob", "classification": "SOURCE-DEFINES / observed values exceed it",
     "documented_meaning": "Caption, verbatim: \"The column 'Rob' include if the telescope is "
                            "robotic or not.\" The caption implies a yes/no concept, but the actual "
                            "column carries more raw values than that -- see the census below."},
]
pd.DataFrame(semantics_rows)


,field,classification,documented_meaning
0,Name,SOURCE-DEFINES,The telescope/instrument identifier used throughout the network table.
1,Location,SOURCE-DEFINES,"Observatory name (free text), not coordinates."
2,MNight,SOURCE-DEFINES,"Caption, verbatim: ""'MNight' corresponds to the time zone of maximum night time."" A UTC scheduling window (e.g. '10-..."
3,Size,SOURCE-DEFINES,Telescope aperture in metres (column header '(m)'). '-' for the spectroscopy-section rows that report no aperture.
4,FOV,SOURCE-DEFINES,"Field of view in degrees (column header '(deg)'), width x height or a single circular value. '-' throughout the spec..."
5,Filter,SOURCE-DEFINES / partially INFERRED,Photometric filter codes for the Photometry-section rows (e.g. 'BVRCIC'; SOURCE-DEFINES). For the Spectroscopy-secti...
6,Mlim,SOURCE-DEFINES,"Caption, verbatim: ""Mlim corresponds to the typical maximum limiting mag obtained in less than 1 h."" A typical/nomin..."
7,Use,SOURCE-DEFINES,"Caption, verbatim: ""The 'Use' colon include 'VF' for very frequent observations for GRANDMA, 'F' as Frequent, 'R' as..."
8,Rob,SOURCE-DEFINES / observed values exceed it,"Caption, verbatim: ""The column 'Rob' include if the telescope is robotic or not."" The caption implies a yes/no conce..."


In [5]:
print(f'Document\'s own temporal-scope statement (page 13 body text, not the caption): '
      f'"{grandma_source["relevant_table_temporal_scope_statement"]}"')
print()
print("Is the table telescope-level or instrument-level? Both sections are nominally per-telescope, "
      "but 2 Photometry rows (CFHT/WIRCam, CFHT/MegaCam) and their Spectroscopy-section counterparts "
      "at other observatories (e.g. GMG-2.4/YFOSC alongside the separate GMG-2.4 photometry row) show "
      "the SAME physical telescope listed once per instrument/mode -- confirmed structurally against "
      "ICARE below. So, as with any such table, a value must not be assumed telescope-wide without "
      "checking which specific row (and therefore instrument) it came from.")


Document's own temporal-scope statement (page 13 body text, not the caption): "The full list of GRANDMA telescopes in 2026 is presented in Table 1.1."

Is the table telescope-level or instrument-level? Both sections are nominally per-telescope, but 2 Photometry rows (CFHT/WIRCam, CFHT/MegaCam) and their Spectroscopy-section counterparts at other observatories (e.g. GMG-2.4/YFOSC alongside the separate GMG-2.4 photometry row) show the SAME physical telescope listed once per instrument/mode -- confirmed structurally against ICARE below. So, as with any such table, a value must not be assumed telescope-wide without checking which specific row (and therefore instrument) it came from.


## GRANDMA table census

Per-column census of the 9 real data columns (39 rows: 32 Photometry +
7 Spectroscopy). No normalization; raw text preserved, including known
PDF glyph-encoding artifacts (a raised prime mark apparently rendered as
a literal "1" in filter codes, "^" standing in for "x" in FOV values).


In [6]:
DATA_COLUMNS = ["telescope_name", "location", "mnight_h", "size_m", "fov_deg", "filter", "mlim",
               "use", "rob"]

grandma_census_rows = []
for column in DATA_COLUMNS:
    series = GRANDMA[column]
    content = has_content(series)
    present = series[content].astype(str)
    grandma_census_rows.append({
        "column": column, "content_count": int(content.sum()),
        "coverage_pct": round(100 * content.sum() / len(GRANDMA), 1),
        "distinct": int(present.nunique()),
        "representative_values": " | ".join(clip(v) for v in present.drop_duplicates().head(4)),
    })
pd.DataFrame(grandma_census_rows)


,column,content_count,coverage_pct,distinct,representative_values
0,telescope_name,39,100.0,38,TRT-SBO | TNT | Zadko | Xinglong-2.16
1,location,39,100.0,26,Spring Brook Obs. | Thai Nat Obs. | Xinglong Obs. | Gingin Obs.
2,mnight_h,39,100.0,15,10–16h | 11–23h | 12–22h | 14–00h
3,size_m,34,87.2,19,0.70 | 2.40 | 0.80 | 1.00
4,fov_deg,32,82.1,24,0.17 ˆ 0.17 | 0.13 ˆ 0.13 | 0.19 ˆ 0.19 | 0.17 ˆ 0.12
5,filter,39,100.0,24,UBV R I C C | u1g1r1i1z1 | g1r1i1 BV | g1r1i1I C
6,mlim,39,100.0,16,21.5 | 24 | 21 | 22
7,use,39,100.0,8,F | VR | BV R | nOP
8,rob,39,100.0,6,yes | no | remote | np


In [7]:
print("Use: raw value counts vs the caption's documented vocabulary (VF/F/R/VR/nOP):")
print(GRANDMA["use"].value_counts().to_string())
outside_vocab = GRANDMA.loc[~GRANDMA["use"].isin({"VF", "F", "R", "VR", "nOP"}),
                            ["row_index_in_source", "telescope_name", "use", "filter"]]
print(f"\n{len(outside_vocab)} row(s) carry a 'use' value outside that vocabulary:")
print(outside_vocab.to_string(index=False))
print("Row 2 ('BV R') looks like a column-boundary extraction ambiguity flagged by the extractor "
      "itself; rows 27 ('O') and 34 ('nOp') are raw values/case variants actually printed in the "
      "source (not an extraction artifact) -- 'nOp' vs 'nOP' elsewhere is a genuine source case "
      "variant, left unchanged.")


Use: raw value counts vs the caption's documented vocabulary (VF/F/R/VR/nOP):
use
VR      13
F        8
R        7
VF       5
nOP      3
BV R     1
O        1
nOp      1

3 row(s) carry a 'use' value outside that vocabulary:
 row_index_in_source    telescope_name  use      filter
                   2               TNT BV R   g1r1i1 BV
                  27 Perkin-Elmer Tel.    O UBV R I C C
                  34  Terskol- 2m/MMCS  nOp 3800 ´ 9000
Row 2 ('BV R') looks like a column-boundary extraction ambiguity flagged by the extractor itself; rows 27 ('O') and 34 ('nOp') are raw values/case variants actually printed in the source (not an extraction artifact) -- 'nOp' vs 'nOP' elsewhere is a genuine source case variant, left unchanged.


In [8]:
print("Rob: raw value counts (the caption only documents a yes/no concept):")
print(GRANDMA["rob"].value_counts().to_string())
print()
print("distinct filter/Mlim column values on the Spectroscopy rows (wavelength ranges, not filter "
      "codes):")
print(GRANDMA.loc[GRANDMA["section"] == "Spectroscopy", ["telescope_name", "filter", "mlim"]]
      .to_string(index=False))


Rob: raw value counts (the caption only documents a yes/no concept):
rob
no        28
yes        7
remote     1
np         1
semi       1
nOP        1

distinct filter/Mlim column values on the Spectroscopy rows (wavelength ranges, not filter codes):
      telescope_name                  filter mlim
   2.2m CAHA/- CAFOS 3200 ´ 7000{6300´ 11000   20
            ShAO/T2m             3800 ´ 8000   17
    Terskol- 2m/MMCS             3800 ´ 9000   17
      GMG- 2.4/YFOSC             3400 ´ 9100   19
Xinglong- 2.16/BFOSC             3600 ´ 9600   18
    10.4m GTC/OSIRIS             3630 ´ 7500 22.5
      10.4m GTC/EMIR             890 ´ 13310 20.5


## String and raw-value quality (descriptive; nothing normalized)

Anomalies worth carrying into the decisions stage, found generically
rather than assumed.


In [9]:
duplicate_names = GRANDMA["telescope_name"].value_counts()
duplicate_names = duplicate_names[duplicate_names > 1]
print(f"duplicate telescope_name values in the GRANDMA table: {len(duplicate_names)}")
if len(duplicate_names):
    for name in duplicate_names.index:
        rows = GRANDMA.loc[GRANDMA["telescope_name"] == name,
                           ["row_index_in_source", "location", "mnight_h", "size_m", "section"]]
        print(f"\n  '{name}' appears {duplicate_names[name]} times, at different observatories/sizes:")
        print(rows.to_string(index=False))

wrapped_names = GRANDMA.loc[GRANDMA["telescope_name"].str.contains(r"- \S|\s{2,}", regex=True),
                            ["row_index_in_source", "telescope_name"]]
print(f"\ntelescope_name values that look like a line-wrapped hyphen artifact "
      f"(e.g. 'Terskol- 2m/MMCS' instead of 'Terskol-2m/MMCS'): {len(wrapped_names)}")
print(wrapped_names.to_string(index=False))

print("Representative raw filter codes containing a literal '1' where a prime mark ( ' ) is the "
      "domain-conventional character (e.g. SDSS g'r'i'z' filters), preserved as extracted:")
print(sorted({v for v in GRANDMA["filter"] if "1" in v})[:8])


duplicate telescope_name values in the GRANDMA table: 1

  'TNT' appears 2 times, at different observatories/sizes:
 row_index_in_source      location mnight_h size_m    section
                   1 Thai Nat Obs.   11–23h   2.40 Photometry
                   2 Xinglong Obs.   12–22h   0.80 Photometry

telescope_name values that look like a line-wrapped hyphen artifact (e.g. 'Terskol- 2m/MMCS' instead of 'Terskol-2m/MMCS'): 4
 row_index_in_source       telescope_name
                  32    2.2m CAHA/- CAFOS
                  34     Terskol- 2m/MMCS
                  35       GMG- 2.4/YFOSC
                  36 Xinglong- 2.16/BFOSC
Representative raw filter codes containing a literal '1' where a prime mark ( ' ) is the domain-conventional character (e.g. SDSS g'r'i'z' filters), preserved as extracted:
['3200 ´ 7000{6300´ 11000', '3400 ´ 9100', '890 ´ 13310', 'BVRI g1r1i1z1', 'C, g1r1', 'C, g1r1i1', 'Cg1r1i1', 'g1r1i1 BV']


## Historical / version scope of the GRANDMA reference

Whether Table 1.1 should be read as current, real-time state or as a
fixed snapshot.


In [10]:
print(f"Document type    : {grandma_source['document_type']}")
print(f"Institution      : {grandma_source['institution']}")
print(f"Defended         : {grandma_source['defense_location']}, {grandma_source['defense_date_day_month']}"
      f" -- year: {grandma_source['defense_year']}")
print(f"Table 1.1 itself : \"{grandma_source['relevant_table_temporal_scope_statement']}\"")
print()
print("CONCLUSION: the document explicitly anchors Table 1.1 to a stated year ('in 2026') in its "
      "body text, even though the exact defense date's year is not printed on the title page. This "
      "is the most current GRANDMA network snapshot available to this project. It is still a fixed "
      "manuscript table, not a live feed: 'Use' and 'Rob' describe typical/documented behaviour "
      "('VF'/'F'/'R'/'VR'/'nOP' as a usage-frequency category, robotic/remote/semi as an operating "
      "mode), not a real-time status. In particular, 'nOP' ('not Operational') appearing on 4 rows "
      "here is the source's own documented usage-frequency category, not evidence about whether "
      "that telescope is operational today; it is a manuscript snapshot, not a live-availability "
      "signal, consistent with Stage 1's scoping decision that real-time availability is out of scope.")


Document type    : Habilitation à diriger des recherches (HDR) manuscript
Institution      : Université Paris-Saclay
Defended         : Orsay, 9 October -- year: NOT ESTABLISHED (not printed on the title page; PDF CreationDate metadata is 2026-07-21 but that reflects file generation, not a stated authorial date)
Table 1.1 itself : "The full list of GRANDMA telescopes in 2026 is presented in Table 1.1."

CONCLUSION: the document explicitly anchors Table 1.1 to a stated year ('in 2026') in its body text, even though the exact defense date's year is not printed on the title page. This is the most current GRANDMA network snapshot available to this project. It is still a fixed manuscript table, not a live feed: 'Use' and 'Rob' describe typical/documented behaviour ('VF'/'F'/'R'/'VR'/'nOP' as a usage-frequency category, robotic/remote/semi as an operating mode), not a real-time status. In particular, 'nOP' ('not Operational') appearing on 4 rows here is the source's own documented usage-freq

## ICARE `morning` / `evening` -- documented semantics

Independent of the GRANDMA reference: established from SkyPortal's own
source code (the platform ICARE runs), pinned to the `v1.4.0` tag --
matching the `"version": "1.4.0"` string every ICARE API response in this
project's captures reports. Re-verified here unchanged from the prior
Stage 4 pass.


In [11]:
SKYPORTAL_VERSION_TAG = "v1.4.0"
SKYPORTAL_SOURCE_URL = (
    f"https://github.com/skyportal/skyportal/blob/{SKYPORTAL_VERSION_TAG}/skyportal/models/telescope.py"
)
print(f"authoritative source: {SKYPORTAL_SOURCE_URL}")
print(f"(matches the ICARE API's reported version: 1.4.0, confirmed in this project's own raw "
      f"captures, e.g. {ICARE_RAW_DIR / 'telescopes.json'})")
print()
print("DOCUMENTED SEMANTICS (quoted from skyportal/models/telescope.py, Telescope.current_time and "
      "Telescope.observer):")
print("""
  @property
  def observer(self):
      # returns an astroplan.Observer for this site, or None if lon/lat/elevation
      # is missing/NaN, or fixed_location is False/None

  def current_time(self, refresh=False):
      if (not self.fixed_location or self.lon is None or self.lat is None
              or self.elevation is None or self.observer is None):
          return {"is_night_astronomical": False, "morning": False, "evening": False}
      # otherwise: morning = next_twilight_morning_astronomical() (astropy Time)
      #            evening = next_twilight_evening_astronomical() (astropy Time)
      #            is_night_astronomical = bool(morning.jd < evening.jd)
""")
print("So: morning/evening are the NEXT astronomical (-18 degree) twilight timestamps at the "
      "telescope's site, computed live via astroplan at request time -- not a stored schedule. "
      "When the site has no fixed/known location, the API returns the literal boolean False for "
      "both fields instead of a timestamp.")


authoritative source: https://github.com/skyportal/skyportal/blob/v1.4.0/skyportal/models/telescope.py
(matches the ICARE API's reported version: 1.4.0, confirmed in this project's own raw captures, e.g. /home/meneses/project_astronomical/MAFORAI/data/raw/telescopes/icare/capture_20260808_071334/telescopes.json)

DOCUMENTED SEMANTICS (quoted from skyportal/models/telescope.py, Telescope.current_time and Telescope.observer):

  @property
  def observer(self):
      # returns an astroplan.Observer for this site, or None if lon/lat/elevation
      # is missing/NaN, or fixed_location is False/None

  def current_time(self, refresh=False):
      if (not self.fixed_location or self.lon is None or self.lat is None
              or self.elevation is None or self.observer is None):
          return {"is_night_astronomical": False, "morning": False, "evening": False}
      # otherwise: morning = next_twilight_morning_astronomical() (astropy Time)
      #            evening = next_twilight_evenin

In [12]:
tel = ICARE["telescopes"]

def morning_category(value):
    if value is None:
        return "null"
    if value == "false":
        return "json-bool false"
    if value == "true":
        return "json-bool true"
    if isinstance(value, str) and value.startswith('"'):
        return "quoted timestamp string"
    return f"other: {value!r}"

observed = pd.DataFrame({
    "morning_category": tel["morning"].map(morning_category),
    "fixed_location": tel["fixed_location"],
    "has_lat": has_content(tel["lat"]),
})
agreement = pd.crosstab(observed["morning_category"], observed["fixed_location"] & observed["has_lat"],
                        rownames=["morning value"], colnames=["fixed_location AND has lat/lon"])
print("ACTUAL OBSERVED REPRESENTATION vs the documented condition:")
print(agreement)
predicted_false = ~(tel["fixed_location"] & has_content(tel["lat"]))
actual_false = tel["morning"] == "false"
agree_count = int((predicted_false == actual_false).sum())
print(f"\nAGREEMENT: the documented condition (not fixed_location OR lon/lat/elevation missing) "
      f"predicts 'False' exactly where it is actually observed on {agree_count} / {len(tel)} telescopes.")
print("This fully explains the mixed representation Stage 3 flagged: it is not a data-quality issue, "
      "it is the documented behaviour of a live-computed field frozen at one capture instant.")

note_question(
    "morning/evening are documented as live-computed 'next twilight from request time', not stored "
    "telescope metadata -- should the final schema keep them as a frozen-capture artifact, drop them, "
    "or replace them with a genuinely static property (e.g. site coordinates) derivable on demand?",
    "SkyPortal v1.4.0 source confirms morning/evening = astroplan-computed next twilight, False "
    "exactly when fixed_location/lon/lat/elevation is missing, agreeing with 89/89 observed rows",
    "telescopes.morning, telescopes.evening, telescopes.fixed_location, telescopes.lat/lon",
    "89 telescopes",
)


ACTUAL OBSERVED REPRESENTATION vs the documented condition:
fixed_location AND has lat/lon  False  True 
morning value                               
json-bool false                     8      0
quoted timestamp string             0     81

AGREEMENT: the documented condition (not fixed_location OR lon/lat/elevation missing) predicts 'False' exactly where it is actually observed on 89 / 89 telescopes.
This fully explains the mixed representation Stage 3 flagged: it is not a data-quality issue, it is the documented behaviour of a live-computed field frozen at one capture instant.


## ICARE allocation semantics

Same method, independent of the GRANDMA reference: the `Allocation` model
in SkyPortal `v1.4.0` source code, plus the specific allocation id 78 /
telescope id 137 discrepancy Stage 3 found. Re-verified here unchanged.


In [13]:
ALLOCATION_SOURCE_URL = (
    f"https://github.com/skyportal/skyportal/blob/{SKYPORTAL_VERSION_TAG}/skyportal/models/allocation.py"
)
print(f"authoritative source: {ALLOCATION_SOURCE_URL}")
print()
print("DOCUMENTED SEMANTICS (quoted class docstring and column doc= strings):")
print('  class Allocation: "An allocation of observing time on a robotic instrument."')
for field, doc in [
    ("pi", "The PI of the allocation's proposal."),
    ("proposal_id", "The ID of the proposal associated with this allocation."),
    ("start_date", "The UTC start date of the allocation."),
    ("end_date", "The UTC end date of the allocation."),
    ("hours_allocated", "The number of hours allocated."),
    ("default_share_group_ids", "List of default group IDs to share data with"),
    ("types", "The type of allocation."),
    ("group_id", "The ID of the Group the allocation is associated with."),
    ("instrument_id", "The ID of the Instrument the allocation is associated with."),
]:
    print(f"    {field:26s}: {doc}")
print()
print("No column or property named 'availability', 'active', 'status', or similar exists on this "
      "model. What the data allow us to claim:")
print("  - AUTHORIZATION/ACCESS : an allocation grants a Group/PI a budget of hours on one "
      "Instrument (pi, group_id, instrument_id, hours_allocated).")
print("  - VALIDITY             : start_date/end_date exist on the model as scalar UTC dates; ICARE's "
      "'validity_ranges' column (always an empty list in this capture) does not correspond to a "
      "column found in this file and may be a schema addition not present in the public v1.4.0 tag, "
      "or a computed/serialized field defined elsewhere -- NOT ESTABLISHED from this source alone.")
print("  - REAL-TIME AVAILABILITY: NOT ESTABLISHED. hours_allocated is a budget, not a live counter; "
      "no field tracks remaining hours, active/inactive state, or current usage.")


authoritative source: https://github.com/skyportal/skyportal/blob/v1.4.0/skyportal/models/allocation.py

DOCUMENTED SEMANTICS (quoted class docstring and column doc= strings):
  class Allocation: "An allocation of observing time on a robotic instrument."
    pi                        : The PI of the allocation's proposal.
    proposal_id               : The ID of the proposal associated with this allocation.
    start_date                : The UTC start date of the allocation.
    end_date                  : The UTC end date of the allocation.
    hours_allocated           : The number of hours allocated.
    default_share_group_ids   : List of default group IDs to share data with
    types                     : The type of allocation.
    group_id                  : The ID of the Group the allocation is associated with.
    instrument_id             : The ID of the Instrument the allocation is associated with.

No column or property named 'availability', 'active', 'status', or similar

In [14]:
ACCESS_CONTROL_POLICY = (
    "create = read = update = delete = (accessible_by_group_members "
    "& AccessibleIfRelatedRowsAreAccessible(instrument='read'))"
)
print(f"Allocation access-control policy (skyportal/models/allocation.py, v1.4.0):\n  {ACCESS_CONTROL_POLICY}")
print()
print("This means every read of the /api/allocation endpoint is scoped to allocations belonging to "
      "groups the requesting token's user is a member of. A raw SQLAlchemy relationship traversal "
      "(e.g. serializing telescope.instruments[i].allocations while building the nested /api/telescope "
      "response) is a different code path and is not guaranteed to apply the same group-membership "
      "filter as the dedicated, permission-scoped /api/allocation query.")
print()
print("Discrepancy under investigation: telescope id=137 (Colibri) nests allocation id=78 "
      "(instrument_id=85, group_id=122, PI='Stephane Basa') inside its serialized instruments[0]."
      "allocations, but no row with id=78 exists in the standalone /api/allocation capture.")
print()
print("CANDIDATE EXPLANATION (source-documented, not fully traced to certainty): group-based "
      "permission filtering. Allocation id 78 belongs to group_id=122; if the API token used for this "
      "capture's user is not a member of group 122, the permission-scoped /api/allocation query would "
      "legitimately omit it, while an unfiltered nested-relationship serialization inside /api/telescope "
      "would still include it. This is consistent with the documented access-control policy above and "
      "is the most plausible mechanism found, but the exact telescope/instrument serialization code "
      "path was not traced line-by-line, so this is reported as PLAUSIBLE, not CONFIRMED. Stale nested "
      "relation and endpoint-specific visibility are the same underlying mechanism seen from two "
      "angles; a soft-deleted/inactive allocation was not found as a documented concept on this model "
      "(no such column exists) and is therefore not a candidate explanation here.")


Allocation access-control policy (skyportal/models/allocation.py, v1.4.0):
  create = read = update = delete = (accessible_by_group_members & AccessibleIfRelatedRowsAreAccessible(instrument='read'))

This means every read of the /api/allocation endpoint is scoped to allocations belonging to groups the requesting token's user is a member of. A raw SQLAlchemy relationship traversal (e.g. serializing telescope.instruments[i].allocations while building the nested /api/telescope response) is a different code path and is not guaranteed to apply the same group-membership filter as the dedicated, permission-scoped /api/allocation query.

Discrepancy under investigation: telescope id=137 (Colibri) nests allocation id=78 (instrument_id=85, group_id=122, PI='Stephane Basa') inside its serialized instruments[0].allocations, but no row with id=78 exists in the standalone /api/allocation capture.

CANDIDATE EXPLANATION (source-documented, not fully traced to certainty): group-based permission filter

## ICARE <-> GRANDMA diagnostic name matching (analysis only, not persisted)

Recomputed entirely from scratch against the new 39-row table. Matching is
done PER GRANDMA ROW, not per unique name: the table itself contains a
duplicate telescope_name ('TNT', at two different observatories with two
different apertures), so matching by name alone would silently merge two
different physical telescopes. Comparison-only: nothing here mutates
`ICARE["telescopes"]` or `GRANDMA`.


In [15]:
tel = ICARE["telescopes"]
icare_names = set(tel["name"])
tel_by_name = tel.set_index("name")["diameter"]

def normalize(name):
    return name.strip().casefold()

def tokenize(name):
    return {t.lower() for t in re.split(r"[/\-\s]+", name) if len(t) >= 2}

def token_overlap_score(g_tokens, i_tokens):
    exact = g_tokens & i_tokens
    fuzzy = {gt for gt in g_tokens for it in i_tokens if gt != it and (gt in it or it in gt)}
    return len(exact | fuzzy)

# Known, unambiguous facility acronym actually relevant to this table
# (explicit "known alias" evidence, per the brief's own guidance -- not
# silently inferred, and used only alongside name-token/aperture evidence).
KNOWN_ALIASES = {"CFHT": "Canada-France-Hawaii Telescope"}
MATCH_APERTURE_TOLERANCE_M = 0.05

def parse_size(value):
    value = str(value).strip()
    if value in ("", "-"):
        return None
    match = re.search(r"[\d.]+", value)
    return float(match.group()) if match else None

def aperture_agrees(icare_name, g_size):
    """True/False/None (None = not comparable because a value is missing).
    Rounded before comparing so binary floating-point representation error
    cannot flip a boundary result."""
    icare_dia = tel_by_name.get(icare_name)
    if icare_dia is None or pd.isna(icare_dia) or g_size is None:
        return None
    delta = round(abs(float(icare_dia) - float(g_size)), 3)
    return bool(delta <= MATCH_APERTURE_TOLERANCE_M)

match_rows = []
for _, row in GRANDMA.iterrows():
    g_name = row["telescope_name"]
    g_size = parse_size(row["size_m"])

    if g_name in icare_names:
        match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                           "candidates": [g_name], "classification": "EXACT", "evidence": "raw name match"})
        continue
    if normalize(g_name) in {normalize(n) for n in icare_names} and g_name.strip() != g_name:
        norm_hit = next(n for n in icare_names if normalize(n) == normalize(g_name))
        match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                           "candidates": [norm_hit], "classification": "EXACT (after strip/casefold)",
                           "evidence": "comparison-only whitespace/case difference"})
        continue

    g_tokens = tokenize(g_name)
    scores = {}
    for i_name in icare_names:
        score = token_overlap_score(g_tokens, tokenize(i_name))
        for acronym, expansion in KNOWN_ALIASES.items():
            if acronym.lower() in g_tokens and i_name == expansion:
                score += 2
        if score > 0:
            scores[i_name] = score

    if not scores:
        match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                           "candidates": [], "classification": "UNMATCHED",
                           "evidence": "no name-token overlap found"})
        continue

    max_score = max(scores.values())
    top_candidates = sorted(n for n, s in scores.items() if s == max_score)

    if len(top_candidates) == 1:
        candidate = top_candidates[0]
        agrees = aperture_agrees(candidate, g_size)
        if agrees is False:
            match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                               "candidates": [candidate], "classification": "AMBIGUOUS",
                               "evidence": f"unique name candidate but aperture disagrees "
                                          f"(ICARE={tel_by_name.get(candidate)}, GRANDMA={g_size})"})
        else:
            match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                               "candidates": [candidate], "classification": "HIGH-CONFIDENCE CANDIDATE",
                               "evidence": f"unique name-token candidate (score={max_score}), aperture "
                                          f"{'agrees' if agrees else 'unavailable on one side'} "
                                          f"(ICARE={tel_by_name.get(candidate)}, GRANDMA={g_size})"})
    else:
        agreeing = [c for c in top_candidates if aperture_agrees(c, g_size) is True]
        if len(agreeing) == 1:
            match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                               "candidates": [agreeing[0]], "classification": "HIGH-CONFIDENCE CANDIDATE",
                               "evidence": f"tie at score={max_score} among {top_candidates}, broken by "
                                          f"unique aperture agreement (GRANDMA={g_size})"})
        else:
            match_rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                               "candidates": top_candidates, "classification": "AMBIGUOUS",
                               "evidence": f"{len(top_candidates)}-way tie at score={max_score}, not "
                                          f"resolved by aperture: "
                                          + ", ".join(f"{c} (diameter={tel_by_name.get(c)})" for c in top_candidates)})

matches = pd.DataFrame(match_rows)
matches


,row_index_in_source,grandma_name,candidates,classification,evidence
0,0,TRT-SBO,[Thai Robotic Telescope - SBO],HIGH-CONFIDENCE CANDIDATE,"unique name-token candidate (score=1), aperture agrees (ICARE=0.7, GRANDMA=0.7)"
1,1,TNT,"[UBAI/NT-60, Xinglong-TNT]",AMBIGUOUS,"2-way tie at score=1, not resolved by aperture: UBAI/NT-60 (diameter=0.6), Xinglong-TNT (diameter=0.8)"
2,2,TNT,[Xinglong-TNT],HIGH-CONFIDENCE CANDIDATE,"tie at score=1 among ['UBAI/NT-60', 'Xinglong-TNT'], broken by unique aperture agreement (GRANDMA=0.8)"
3,3,Zadko,[Zadko],EXACT,raw name match
4,4,Xinglong-2.16,[Xinglong-2.16m],HIGH-CONFIDENCE CANDIDATE,"unique name-token candidate (score=2), aperture agrees (ICARE=2.16, GRANDMA=2.16)"
5,5,GMG-2.4,[GMG-2.4m],HIGH-CONFIDENCE CANDIDATE,"unique name-token candidate (score=2), aperture agrees (ICARE=2.4, GRANDMA=2.4)"
6,6,BJP/ALI-50 TNOT,[PicduMidi/T50],HIGH-CONFIDENCE CANDIDATE,"tie at score=1 among ['AbAO-T150', 'OSN/T150', 'PicduMidi/T50', 'TNOT'], broken by unique aperture agreement (GRANDM..."
7,7,UBAI/NT-60,[UBAI/NT-60],EXACT,raw name match
8,8,UBAI/ST-60 NutelA,[UBAI/ST-60],HIGH-CONFIDENCE CANDIDATE,"unique name-token candidate (score=3), aperture agrees (ICARE=0.6, GRANDMA=0.6)"
9,9,TAROT/TRE,[TAROT/TRE],EXACT,raw name match


In [16]:
counts = matches["classification"].value_counts()
print("Diagnostic classification of all 39 GRANDMA rows:")
print(counts.to_string())
print(f"\nOverall: EXACT={int(counts.get('EXACT', 0))}, "
      f"HIGH-CONFIDENCE CANDIDATE={int(counts.get('HIGH-CONFIDENCE CANDIDATE', 0))}, "
      f"AMBIGUOUS={int(counts.get('AMBIGUOUS', 0))}, "
      f"UNMATCHED={int(counts.get('UNMATCHED', 0))}  (total GRANDMA rows={len(GRANDMA)})")
print("\nNone of this is a persisted mapping: 'matches' exists only in this notebook's memory and "
      "is never written to disk.")


Diagnostic classification of all 39 GRANDMA rows:
classification
HIGH-CONFIDENCE CANDIDATE    23
EXACT                        10
AMBIGUOUS                     3
UNMATCHED                     3

Overall: EXACT=10, HIGH-CONFIDENCE CANDIDATE=23, AMBIGUOUS=3, UNMATCHED=3  (total GRANDMA rows=39)

None of this is a persisted mapping: 'matches' exists only in this notebook's memory and is never written to disk.


## Potential complementary coverage (descriptive only; nothing is enriched)

For EXACT and HIGH-CONFIDENCE CANDIDATE matches only, how much of ICARE's
sparse footprint/sensitivity/filter/robotic information GRANDMA *could*
speak to -- not applied here. Unlike the previously (incorrectly) used
reference, this document DOES carry a Rob column, so robotic metadata is
now a genuinely comparable category, not just a "missing on one side" gap.


In [17]:
matched = matches[matches["classification"].isin(["EXACT", "EXACT (after strip/casefold)",
                                                   "HIGH-CONFIDENCE CANDIDATE"])].copy()
matched["icare_name"] = matched["candidates"].map(lambda c: c[0])
print(f"Matched GRANDMA rows considered (EXACT + HIGH-CONFIDENCE CANDIDATE): {len(matched)}")

inst = ICARE["instruments"]

def icare_instruments_for(icare_telescope_name):
    telescope_id = tel.loc[tel["name"] == icare_telescope_name, "id"]
    if telescope_id.empty:
        return inst.iloc[0:0]
    return inst[inst["telescope_id"] == telescope_id.iloc[0]]

coverage_rows = []
for _, m in matched.iterrows():
    g_row = GRANDMA[GRANDMA["row_index_in_source"] == m["row_index_in_source"]].iloc[0]
    matched_instruments = icare_instruments_for(m["icare_name"])
    coverage_rows.append({
        "grandma_telescope": g_row["telescope_name"], "icare_telescope": m["icare_name"],
        "icare_instruments_matched": len(matched_instruments),
        "icare_instruments_lacking_region": int((~has_content(matched_instruments["region"])).sum()),
        "icare_instruments_lacking_sensitivity": int(
            (~matched_instruments[[c for c in matched_instruments.columns if c.startswith("sensitivity_data.")]]
             .apply(has_content).any(axis=1)).sum()
        ) if len(matched_instruments) else 0,
        "grandma_has_fov": bool(has_content(pd.Series([g_row["fov_deg"]])).iloc[0]),
        "grandma_has_mlim": bool(has_content(pd.Series([g_row["mlim"]])).iloc[0]),
        "grandma_has_filter": bool(has_content(pd.Series([g_row["filter"]])).iloc[0]),
        "grandma_has_rob": bool(has_content(pd.Series([g_row["rob"]])).iloc[0]),
    })
coverage = pd.DataFrame(coverage_rows)
coverage


Matched GRANDMA rows considered (EXACT + HIGH-CONFIDENCE CANDIDATE): 33


,grandma_telescope,icare_telescope,icare_instruments_matched,icare_instruments_lacking_region,icare_instruments_lacking_sensitivity,grandma_has_fov,grandma_has_mlim,grandma_has_filter,grandma_has_rob
0,TRT-SBO,Thai Robotic Telescope - SBO,1,1,1,True,True,True,True
1,TNT,Xinglong-TNT,1,1,1,True,True,True,True
2,Zadko,Zadko,1,1,1,True,True,True,True
3,Xinglong-2.16,Xinglong-2.16m,2,2,2,True,True,True,True
4,GMG-2.4,GMG-2.4m,2,2,2,True,True,True,True
5,BJP/ALI-50 TNOT,PicduMidi/T50,1,1,1,True,True,True,True
6,UBAI/NT-60,UBAI/NT-60,1,1,1,True,True,True,True
7,UBAI/ST-60 NutelA,UBAI/ST-60,1,1,1,True,True,True,True
8,TAROT/TRE,TAROT/TRE,1,0,0,True,True,True,True
9,Les Makes/T60,Les-Makes/T60,1,0,1,True,True,True,True


In [18]:
summary_rows = [
    {"category": "FOV / footprint",
     "icare_instruments_lacking": int(coverage["icare_instruments_lacking_region"].sum()),
     "matched_grandma_rows_carrying_it": int(coverage["grandma_has_fov"].sum()),
     "of_matched_rows": len(coverage)},
    {"category": "Mlim / sensitivity evidence",
     "icare_instruments_lacking": int(coverage["icare_instruments_lacking_sensitivity"].sum()),
     "matched_grandma_rows_carrying_it": int(coverage["grandma_has_mlim"].sum()),
     "of_matched_rows": len(coverage)},
    {"category": "filters",
     "icare_instruments_lacking": int((~has_content(inst.loc[inst["telescope_id"].isin(
         tel.loc[tel["name"].isin(matched["icare_name"]), "id"]), "filters"]
         .map(lambda v: v != "[]"))).sum()),
     "matched_grandma_rows_carrying_it": int(coverage["grandma_has_filter"].sum()),
     "of_matched_rows": len(coverage)},
    {"category": "aperture/size",
     "icare_instruments_lacking": int((~has_content(tel.loc[tel["name"].isin(
         matched["icare_name"]), "diameter"])).sum()),
     "matched_grandma_rows_carrying_it": len(coverage),
     "of_matched_rows": len(coverage)},
    {"category": "robotic/remote metadata",
     "icare_instruments_lacking": "n/a -- ICARE.telescopes.robotic is 100% populated (see A_eda)",
     "matched_grandma_rows_carrying_it": int(coverage["grandma_has_rob"].sum()),
     "of_matched_rows": len(coverage)},
]
print(f"Potential coverage gain, matched pairs only (n={len(coverage)}):")
pd.DataFrame(summary_rows)


Potential coverage gain, matched pairs only (n=33):


,category,icare_instruments_lacking,matched_grandma_rows_carrying_it,of_matched_rows
0,FOV / footprint,34,29,33
1,Mlim / sensitivity evidence,38,33,33
2,filters,0,33,33
3,aperture/size,0,33,33
4,robotic/remote metadata,n/a -- ICARE.telescopes.robotic is 100% populated (see A_eda),33,33


## Conflict analysis (descriptive; no resolution, no precedence)

Where both sources carry apparently corresponding information, for the
same matched pairs. `AGREES` / `DIFFERS` / `NOT DIRECTLY COMPARABLE` /
`MISSING ON ONE SIDE`. Recomputed from scratch; no numbers carried over
from any previous pass.


In [19]:
CONFLICT_APERTURE_TOLERANCE_M = 0.005
conflict_rows = []
for _, m in matched.iterrows():
    g_row = GRANDMA[GRANDMA["row_index_in_source"] == m["row_index_in_source"]].iloc[0]
    icare_dia = tel.loc[tel["name"] == m["icare_name"], "diameter"]
    icare_dia = icare_dia.iloc[0] if len(icare_dia) else None
    g_size = parse_size(g_row["size_m"])
    if icare_dia is None or g_size is None:
        status, detail = "MISSING ON ONE SIDE", f"icare={icare_dia}, grandma={g_size}"
    else:
        delta = round(abs(icare_dia - g_size), 3)
        status = "AGREES" if delta <= CONFLICT_APERTURE_TOLERANCE_M else "DIFFERS"
        detail = f"icare={icare_dia}, grandma={g_size}, delta={delta}"
    conflict_rows.append({"icare_telescope": m["icare_name"], "grandma_telescope": g_row["telescope_name"],
                          "field_pair": "diameter (ICARE) vs Size (GRANDMA)", "status": status, "detail": detail})

conflicts = pd.DataFrame(conflict_rows)
conflicts


,icare_telescope,grandma_telescope,field_pair,status,detail
0,Thai Robotic Telescope - SBO,TRT-SBO,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.7, grandma=0.7, delta=0.0"
1,Xinglong-TNT,TNT,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.8, grandma=0.8, delta=0.0"
2,Zadko,Zadko,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=1.0, grandma=1.0, delta=0.0"
3,Xinglong-2.16m,Xinglong-2.16,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=2.16, grandma=2.16, delta=0.0"
4,GMG-2.4m,GMG-2.4,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=2.4, grandma=2.4, delta=0.0"
5,PicduMidi/T50,BJP/ALI-50 TNOT,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.5, grandma=0.5, delta=0.0"
6,UBAI/NT-60,UBAI/NT-60,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.6, grandma=0.6, delta=0.0"
7,UBAI/ST-60,UBAI/ST-60 NutelA,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.6, grandma=0.6, delta=0.0"
8,TAROT/TRE,TAROT/TRE,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.18, grandma=0.18, delta=0.0"
9,Les-Makes/T60,Les Makes/T60,diameter (ICARE) vs Size (GRANDMA),AGREES,"icare=0.6, grandma=0.6, delta=0.0"


In [20]:
print(conflicts["status"].value_counts().to_string())
disagreeing = conflicts[conflicts["status"] == "DIFFERS"]
if len(disagreeing):
    print(f"\n{len(disagreeing)} pair(s) DIFFER beyond {CONFLICT_APERTURE_TOLERANCE_M} m tolerance:")
    print(disagreeing.to_string(index=False))
    note_question(
        "Where ICARE diameter and GRANDMA Size disagree by a small but non-zero amount, which value "
        "should the final schema prefer, if either?",
        f"{len(disagreeing)} of {len(conflicts)} matched pairs differ beyond {CONFLICT_APERTURE_TOLERANCE_M} m",
        "telescopes.diameter (ICARE), size_m (GRANDMA)",
        f"{len(disagreeing)} telescope(s)",
    )
else:
    print("\nno pair differs beyond tolerance")


status
AGREES                 27
MISSING ON ONE SIDE     4
DIFFERS                 2

2 pair(s) DIFFER beyond 0.005 m tolerance:
               icare_telescope grandma_telescope                         field_pair  status                              detail
Canada-France-Hawaii Telescope       CFHT/WIRCam diameter (ICARE) vs Size (GRANDMA) DIFFERS icare=3.58, grandma=3.6, delta=0.02
Canada-France-Hawaii Telescope      CFHT/MegaCam diameter (ICARE) vs Size (GRANDMA) DIFFERS icare=3.58, grandma=3.6, delta=0.02


In [21]:
rob_boolean_like = GRANDMA["rob"].isin(["yes", "no"])
rob_rows = []
for _, m in matched.iterrows():
    g_row = GRANDMA[GRANDMA["row_index_in_source"] == m["row_index_in_source"]].iloc[0]
    icare_robotic = tel.loc[tel["name"] == m["icare_name"], "robotic"]
    icare_robotic = bool(icare_robotic.iloc[0]) if len(icare_robotic) else None
    g_rob = g_row["rob"]
    if g_rob not in ("yes", "no"):
        status, detail = "NOT DIRECTLY COMPARABLE", f"grandma rob={g_rob!r} is not a plain yes/no value"
    elif icare_robotic is None:
        status, detail = "MISSING ON ONE SIDE", "icare robotic missing"
    else:
        agrees = (g_rob == "yes") == icare_robotic
        status, detail = ("AGREES" if agrees else "DIFFERS"), f"icare={icare_robotic}, grandma={g_rob!r}"
    rob_rows.append({"icare_telescope": m["icare_name"], "grandma_telescope": g_row["telescope_name"],
                     "field_pair": "robotic (ICARE bool) vs Rob (GRANDMA text)", "status": status, "detail": detail})

rob_conflicts = pd.DataFrame(rob_rows)
print(rob_conflicts["status"].value_counts().to_string())
rob_conflicts


status
AGREES                     26
NOT DIRECTLY COMPARABLE     4
DIFFERS                     3


,icare_telescope,grandma_telescope,field_pair,status,detail
0,Thai Robotic Telescope - SBO,TRT-SBO,robotic (ICARE bool) vs Rob (GRANDMA text),DIFFERS,"icare=False, grandma='yes'"
1,Xinglong-TNT,TNT,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
2,Zadko,Zadko,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
3,Xinglong-2.16m,Xinglong-2.16,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
4,GMG-2.4m,GMG-2.4,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
5,PicduMidi/T50,BJP/ALI-50 TNOT,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
6,UBAI/NT-60,UBAI/NT-60,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
7,UBAI/ST-60,UBAI/ST-60 NutelA,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=False, grandma='no'"
8,TAROT/TRE,TAROT/TRE,robotic (ICARE bool) vs Rob (GRANDMA text),AGREES,"icare=True, grandma='yes'"
9,Les-Makes/T60,Les Makes/T60,robotic (ICARE bool) vs Rob (GRANDMA text),NOT DIRECTLY COMPARABLE,grandma rob='remote' is not a plain yes/no value


In [22]:
print("Filters: ICARE instrument-level JSON list of short codes (e.g. 'bessellv', 'ps1::open') vs "
      "GRANDMA telescope-level free text (e.g. 'BVRCIC'). Different naming CONVENTIONS entirely "
      "(letter-per-band vs prefixed system codes) -- classified NOT DIRECTLY COMPARABLE here; no "
      "attempt is made to assert the plausible Johnson-Cousins <-> Bessell correspondence without "
      "explicit evidence for it.")
print()
print("Location: GRANDMA carries a free-text observatory name; ICARE carries numeric lat/lon/elevation. "
      "Different REPRESENTATIONS, not directly comparable without external geocoding, which is out of "
      "scope for this notebook -- classified NOT DIRECTLY COMPARABLE.")
print()
disagreeing_rob = rob_conflicts[rob_conflicts["status"] == "DIFFERS"]
if len(disagreeing_rob):
    note_question(
        "GRANDMA's Rob column disagrees with ICARE's robotic flag on some matched telescopes, or "
        "carries values (remote/semi/np) that a plain boolean cannot represent -- how should robotic "
        "status be represented if both sources are ever combined?",
        f"{len(disagreeing_rob)} of {len(rob_conflicts)} matched pairs DIFFER; "
        f"{int((rob_conflicts['status'] == 'NOT DIRECTLY COMPARABLE').sum())} more are not "
        f"comparable at all because GRANDMA's Rob carries more than yes/no",
        "telescopes.robotic (ICARE), rob (GRANDMA)",
        f"{len(rob_conflicts)} matched telescopes",
    )


Filters: ICARE instrument-level JSON list of short codes (e.g. 'bessellv', 'ps1::open') vs GRANDMA telescope-level free text (e.g. 'BVRCIC'). Different naming CONVENTIONS entirely (letter-per-band vs prefixed system codes) -- classified NOT DIRECTLY COMPARABLE here; no attempt is made to assert the plausible Johnson-Cousins <-> Bessell correspondence without explicit evidence for it.

Location: GRANDMA carries a free-text observatory name; ICARE carries numeric lat/lon/elevation. Different REPRESENTATIONS, not directly comparable without external geocoding, which is out of scope for this notebook -- classified NOT DIRECTLY COMPARABLE.



## Consolidated candidate questions for B_decisions

Combining `A_eda.ipynb`'s carried-over questions with what this
(corrected) notebook adds. Not answered here.


In [23]:
# Carried over from A_eda.ipynb (restated with this notebook's confirming evidence where relevant).
note_question(
    "Do raw `band`/filter labels require a canonical representation before they can be grouped or "
    "compared, and should GRANDMA's differently-conventioned filter codes (e.g. 'BVRCIC') factor "
    "into that canonicalization at all?",
    "instruments.band has a major 'Optical'/'optical' case-variant pair (90/95 instruments, A_eda); "
    "GRANDMA filters use an unrelated naming convention, confirmed NOT DIRECTLY COMPARABLE above",
    "instruments.band, instruments.filters, observations.filt, GRANDMA.filter",
    "90 of 95 ICARE instruments; 39 GRANDMA rows",
)
note_question(
    "Is ICARE's footprint/region information, together with GRANDMA's telescope-level FOV figures, "
    "sufficient for the intended telescope-resource layer, or does the sparsity on the ICARE side "
    "motivate a different source for this specific field?",
    "ICARE region/footprint populated on 19/95 instruments (A_eda); GRANDMA supplies a numeric FOV "
    "for every Photometry-section row, but only for the (mostly small) subset of ICARE telescopes "
    "matched above, and only at telescope- not instrument-level",
    "instruments.region, instruments.region_summary, GRANDMA.fov_deg",
    "19 of 95 ICARE instruments; up to 33 GRANDMA-matched telescopes",
)
note_question(
    "Is ICARE sensitivity_data (2 of 95 instruments) or GRANDMA's Mlim (a 'typical maximum' depth "
    "under a <1h exposure ceiling, for matched telescopes) the more usable source of limiting-"
    "magnitude evidence, and are the two even measuring the same thing?",
    "ICARE sensitivity_data.* populated on 2/95 instruments (A_eda); GRANDMA's Mlim is SOURCE-DEFINES "
    "for a specific exposure ceiling, not confirmed to match ICARE's exposure/filter context",
    "instruments.sensitivity_data.*, GRANDMA.mlim",
    "2 of 95 ICARE instruments; up to 33 GRANDMA-matched telescopes",
)
note_question(
    "What does an ICARE allocation allow us to claim about resource access, now that the model is "
    "documented as an hours budget with group-scoped read access and no availability field -- and "
    "should the allocation id 78 / telescope id 137 nested-vs-standalone discrepancy be treated as an "
    "acquisition artifact (Stage 1 rerun with broader group membership) or a genuine platform behaviour "
    "to design around?",
    "SkyPortal v1.4.0 Allocation model confirms no availability/active field exists; access is scoped "
    "by accessible_by_group_members; 1 nested allocation (id=78, group_id=122) is absent from the "
    "standalone capture",
    "allocations.hours_allocated, allocations.group_id, telescopes.allocations (nested)",
    "38 allocations; 1 discrepant nested reference",
)
note_question(
    "The GRANDMA<->ICARE HIGH-CONFIDENCE CANDIDATE name pairs found here differ by naming convention "
    "(separator character, unit suffix, location prefix/acronym), and a few were only resolved by a "
    "coincidental aperture tie-break (e.g. 'BJP/ALI-50 TNOT' vs 'PicduMidi/T50' -- same size, "
    "unrelated observatories). Should ICARE telescope names get a formal alias table for cross-source "
    "identity, and should any tie-broken-only candidate found here be reviewed by a person before use?",
    f"{int((matches['classification'] == 'HIGH-CONFIDENCE CANDIDATE').sum())} candidate pairs found "
    f"by name-token overlap + aperture corroboration, 0 of them an exact string match; at least one "
    f"was only resolved via a same-size coincidence between unrelated observatories",
    "telescopes.name, GRANDMA.telescope_name",
    f"{int((matches['classification'] == 'HIGH-CONFIDENCE CANDIDATE').sum())} telescopes",
)

questions = pd.DataFrame(DECISION_QUESTIONS)
print(f"{len(questions)} candidate questions recorded")
questions


8 candidate questions recorded


,question,evidence,relevant_table_fields,measured_scope
0,"morning/evening are documented as live-computed 'next twilight from request time', not stored telescope metadata -- ...","SkyPortal v1.4.0 source confirms morning/evening = astroplan-computed next twilight, False exactly when fixed_locati...","telescopes.morning, telescopes.evening, telescopes.fixed_location, telescopes.lat/lon",89 telescopes
1,"Where ICARE diameter and GRANDMA Size disagree by a small but non-zero amount, which value should the final schema p...",2 of 33 matched pairs differ beyond 0.005 m,"telescopes.diameter (ICARE), size_m (GRANDMA)",2 telescope(s)
2,"GRANDMA's Rob column disagrees with ICARE's robotic flag on some matched telescopes, or carries values (remote/semi/...",3 of 33 matched pairs DIFFER; 4 more are not comparable at all because GRANDMA's Rob carries more than yes/no,"telescopes.robotic (ICARE), rob (GRANDMA)",33 matched telescopes
3,"Do raw `band`/filter labels require a canonical representation before they can be grouped or compared, and should GR...","instruments.band has a major 'Optical'/'optical' case-variant pair (90/95 instruments, A_eda); GRANDMA filters use a...","instruments.band, instruments.filters, observations.filt, GRANDMA.filter",90 of 95 ICARE instruments; 39 GRANDMA rows
4,"Is ICARE's footprint/region information, together with GRANDMA's telescope-level FOV figures, sufficient for the int...",ICARE region/footprint populated on 19/95 instruments (A_eda); GRANDMA supplies a numeric FOV for every Photometry-s...,"instruments.region, instruments.region_summary, GRANDMA.fov_deg",19 of 95 ICARE instruments; up to 33 GRANDMA-matched telescopes
5,Is ICARE sensitivity_data (2 of 95 instruments) or GRANDMA's Mlim (a 'typical maximum' depth under a <1h exposure ce...,ICARE sensitivity_data.* populated on 2/95 instruments (A_eda); GRANDMA's Mlim is SOURCE-DEFINES for a specific expo...,"instruments.sensitivity_data.*, GRANDMA.mlim",2 of 95 ICARE instruments; up to 33 GRANDMA-matched telescopes
6,"What does an ICARE allocation allow us to claim about resource access, now that the model is documented as an hours ...",SkyPortal v1.4.0 Allocation model confirms no availability/active field exists; access is scoped by accessible_by_gr...,"allocations.hours_allocated, allocations.group_id, telescopes.allocations (nested)",38 allocations; 1 discrepant nested reference
7,The GRANDMA<->ICARE HIGH-CONFIDENCE CANDIDATE name pairs found here differ by naming convention (separator character...,"23 candidate pairs found by name-token overlap + aperture corroboration, 0 of them an exact string match; at least o...","telescopes.name, GRANDMA.telescope_name",23 telescopes


## Repository-safety verification

Re-hash every ICARE raw/interim file watched in the setup cell; this
notebook must not have changed any of them.


In [24]:
hashes_after = {str(p): sha256_of(p) for p in ICARE_FILES_TO_WATCH}
changed = [p for p in HASHES_BEFORE if HASHES_BEFORE[p] != hashes_after.get(p)]
print(f"ICARE raw/interim files watched : {len(HASHES_BEFORE)}")
print(f"changed since notebook start     : {len(changed)}")
if changed:
    print(changed)
print(f"\nGRANDMA source PDF sha256 still matches its manifest: "
      f"{sha256_of(preserved_pdf) == grandma_source['sha256']}")


ICARE raw/interim files watched : 9
changed since notebook start     : 0

GRANDMA source PDF sha256 still matches its manifest: True


## Synthesis

**GRANDMA reference.** Table 1.1 ("GRANDMA telescope network") of Sarah
Antier's Habilitation a diriger des recherches, "Multi-messenger Astronomy
with GRANDMA" (Universite Paris-Saclay), was traced, preserved, and
extracted faithfully (39/39 rows across a Photometry and a Spectroscopy
section, all controls PASS, deterministic spot checks PASS). It carries
every column named in the brief -- Name, Location, MNight, Size, FOV,
Filter, Mlim, Use, Rob -- each SOURCE-DEFINES from the caption, with Rob's
actual raw values (yes/no/remote/semi/np/nOP) exceeding the caption's
stated yes/no concept. The document states in its own body text that the
table represents the network "in 2026"; the exact defense year is not
printed on the title page and is recorded as NOT ESTABLISHED rather than
guessed.

**ICARE ambiguities.** `morning`/`evening` are fully explained by
SkyPortal's own `v1.4.0` source (matching ICARE's reported API version):
live-computed next-twilight timestamps, `False` exactly when a site has no
fixed/known location -- confirmed against all 89 telescopes. Allocation
semantics are documented as an hours budget with group-scoped read access
and no availability concept; the allocation id 78 discrepancy has a
plausible, source-documented explanation (group-based permission
filtering) that was not traced to full certainty and is reported as such.

**Diagnostic matching.** 10 exact ICARE<->GRANDMA telescope-name matches
out of 39 GRANDMA rows (one GRANDMA telescope name, 'TNT', is reused at
two different observatories with two different apertures, so matching was
done per row, not per unique name); the remainder split into
HIGH-CONFIDENCE CANDIDATEs (found by name-token overlap, an explicit CFHT
alias, and aperture corroboration, including tie-breaking), a handful of
AMBIGUOUS rows where evidence did not converge, and a few UNMATCHED rows
(exact counts in the tables above). Where matched, aperture/diameter
agrees almost everywhere; a small number of pairs and the new Rob-vs-
robotic comparison surface real, reported disagreements. GRANDMA's
Spectroscopy-section rows correspond to the same physical telescopes as
several Photometry-section rows (e.g. GMG-2.4 and GMG-2.4/YFOSC), direct
evidence for the brief's warning against assigning a telescope-level value
to every instrument on that telescope without checking which row it came
from.

**What remains open** is listed in the consolidated questions table above:
band/filter canonicalization, how to treat the live-computed
`morning`/`evening` fields, what an allocation can be taken to authorize
and the id-78 discrepancy's ultimate cause, whether footprint/FOV and
sensitivity/Mlim evidence (from either source, both sparse or narrowly
matched) is sufficient as-is, and whether ICARE telescope names deserve a
formal, human-reviewed alias table given how some candidates here were
only resolved by a size coincidence. None of these are answered here; they
are handed to `B_decisions.ipynb`.
